# IPI-persistence — single-model PILOT on Kaggle (Qwen2.5-3B-Instruct)

Runs the whole pipeline (Phase A → states → Phase B → clean → judge → analysis) on **one open-weight
model**, locally on the Kaggle GPU. **No API keys, no cost.**

**Before running — set in the right sidebar:**
- **Accelerator:** GPU T4 x1
- **Internet:** On  (needed to clone the repo and download the model)

**Checkpointing / resuming (important — Kaggle sessions can time out):**
- Everything is written to `/kaggle/working/runs` and every model call is cached by an input hash,
  so the run is **resumable**: re-running skips work already done.
- To resume after a timeout: **Save Version** (commit) so `/kaggle/working` is saved as output → open a
  **new version** and attach this notebook's output as an *input dataset* → the restore cell below copies
  `runs/` back → run again; finished calls come from the cache.
- A `runs_checkpoint.zip` is written after every step for easy download.

This notebook runs the **pilot** for one open-weight model: 10 attack + 10 benign scenarios, the full
design (2 repetitions, 6 turns, clean controls), no judge. About 1,700-2,000 model calls, roughly 3 hours
for an 8B model on a T4 x2. Run the other model's notebook the same way, then merge the two checkpoints
locally with `python -m scripts.merge_runs` and analyse with `config/pilot_open.yaml`.

In [ ]:
# --- Cell 1: get the repo + restore any previous checkpoint ---
import os, sys, subprocess, glob, shutil

REPO_URL  = "https://github.com/kritimathurug23-cpu/ipi-persistence.git"
REPO_DIR  = "/kaggle/working/ipi-persistence"
WORK_RUNS = "/kaggle/working/runs"

if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    if r.returncode != 0:
        # Fallback: repo added as a Kaggle dataset (use if the GitHub repo is private)
        cands = glob.glob("/kaggle/input/*/ipi-persistence") + glob.glob("/kaggle/input/*/*/ipi-persistence")
        assert cands, "Clone failed and no repo dataset found. Enable Internet, or add the repo as a dataset."
        shutil.copytree(cands[0], REPO_DIR)
        print("Copied repo from", cands[0])
print("Repo:", REPO_DIR, "->", sorted(os.listdir(REPO_DIR))[:10])

# Restore a checkpoint from a previously-saved output attached as an input dataset
if not os.path.isdir(WORK_RUNS):
    restored = False
    for cand in glob.glob("/kaggle/input/*/runs") + glob.glob("/kaggle/input/*/*/runs"):
        shutil.copytree(cand, WORK_RUNS); print("Restored checkpoint from", cand); restored = True; break
    if not restored:
        print("No prior checkpoint found - starting fresh.")
else:
    print("Existing checkpoint in", WORK_RUNS)

In [ ]:
# --- Cell 2: dependencies (torch is preinstalled on Kaggle GPU) ---
!pip -q install pyyaml pandas numpy scipy statsmodels matplotlib lifelines langdetect scikit-learn tabulate
!pip -q install -U "transformers>=4.44" accelerate
print("dependencies ready")

In [ ]:
# --- Cell 3: load the model and route provider 'local_hf' to it (no repo edits, no API keys) ---
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # ungated; swap for any HF instruct model that fits the GPU
print("Loading", MODEL_ID, "...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
try:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto").eval()
except TypeError:  # older transformers
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

@torch.inference_mode()
def _local_hf(model_id, system, messages, settings):
    msgs = ([{"role": "system", "content": system}] if system else []) + list(messages)
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    max_new = int(settings.get("max_tokens", settings.get("max_new_tokens", 512)))
    gen = dict(max_new_tokens=max_new, pad_token_id=tokenizer.pad_token_id)
    # Default: the model developer's recommended sampling settings (the model's generation_config),
    # as the preregistration specifies. Explicit settings override them; temperature 0 = greedy.
    if "temperature" in settings:
        temp = float(settings["temperature"])
        if temp > 0:
            gen.update(do_sample=True, temperature=temp, top_p=float(settings.get("top_p", 1.0)))
        else:
            gen.update(do_sample=False)
    out = model.generate(**enc, **gen)
    gen_ids = out[0][enc["input_ids"].shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return {"text": text, "usage": {"input": int(enc["input_ids"].shape[1]), "output": int(gen_ids.shape[0])}}

sys.path.insert(0, REPO_DIR)
import harness.providers as P
_orig_chat = P.chat
def _chat(provider, model_id, system, messages, settings, base_url=None, api_key_env=None, **kwargs):
    # **kwargs: the harness also passes options such as prompt_caching; accept and forward them
    if provider == "local_hf":
        return _local_hf(model_id, system, messages, settings)
    return _orig_chat(provider, model_id, system, messages, settings, base_url, api_key_env, **kwargs)
P.chat = _chat                                            # harness.calls calls providers.chat dynamically
print("Default sampling settings:", {k: getattr(model.generation_config, k, None) for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})
print("Model ready; provider 'local_hf' patched in.")

In [ ]:
# --- Cell 4: write a small Kaggle config and smoke-test one generation ---
import yaml
CFG = {
    "run_dir": WORK_RUNS + "/pilot_qwen3b",   # absolute -> lands in persistent /kaggle/working
    "scenarios_dir": "data/scenarios",
    "unrelated_tasks": "data/unrelated_tasks.json",
    "system_prompt": "You are a helpful assistant.",
    "n_subsequent_turns": 6,                    # pilot = full design (proposal Section 21)
    "reps_phase_a": 2, "reps_phase_b": 2, "clean_reps": 3,
    "clean_control_conditions": ["P3"],         # clean-summary control CLEAN_P3, the fair baseline for P3
    "target_states_per_model": 40, "constructed_max_attempts": 3,
    "cross_task_conditions": ["P2", "P3"], "default_task_type": "same_topic",
    "models": [{"name": "qwen3b", "provider": "local_hf", "model_id": MODEL_ID,
                "settings": {"max_tokens": 512}}],
    # No judge in the pilot: the judge must NOT be one of the evaluated models, and there is no API
    # model yet. The harness skips the judge step when judge_model is null.
    "judge_model": None,
}
CFG_PATH = os.path.join(REPO_DIR, "config", "pilot_qwen3b.yaml")
with open(CFG_PATH, "w") as f:
    yaml.safe_dump(CFG, f, sort_keys=False)
print("Wrote", CFG_PATH)
print("Smoke test:", _local_hf(MODEL_ID, "You are a helpful assistant.",
      [{"role": "user", "content": "Say hello in one short sentence."}], {"max_tokens": 32})["text"])

In [ ]:
# --- Cell 5: run the pipeline (resumable). Each step checkpoints; re-running skips finished work. ---
os.chdir(REPO_DIR)
sys.stderr = sys.stdout   # the harness prints progress to stderr; merge so lines appear in order
LIMIT = None  # all 60 attack + 12 benign scenarios. Set LIMIT = 10 for a quick pilot (S001-S010, B001-B010).
# The full benchmark is roughly 11,000-13,000 model calls: about 18-20 hours for an 8B model on a T4 x2,
# so it spans 2-3 Kaggle sessions. Save Version after each session and resume (see the top of the notebook).

from harness.run import Ctx, phase_a, build_states, phase_b, clean, judge
ctx = Ctx(CFG_PATH, models_filter=None, limit=LIMIT)

for name, fn in [("phase-a", phase_a), ("states", build_states),
                 ("phase-b", phase_b), ("clean", clean), ("judge", judge)]:
    print(f"\n===== {name} =====", flush=True)
    fn(ctx)                                              # cached calls + existing rows are skipped
    shutil.make_archive("/kaggle/working/runs_checkpoint", "zip", WORK_RUNS)   # checkpoint after each step
print("\nDone. Checkpoint: /kaggle/working/runs_checkpoint.zip")

In [ ]:
# --- Cell 6: quick results, then the full analysis (analysis only reads results; no model needed) ---
import pandas as pd
from harness.calls import read_jsonl

RES = CFG["run_dir"] + "/results"
pb = pd.DataFrame(read_jsonl(RES + "/phase_b.jsonl"))
print("phase_b rows:", len(pb))
if len(pb):
    d = pb[pb.task_type == "same_topic"]
    print("\nAttacker-marker rate by condition (mean over turns, %):")
    print((d.assign(m=d.marker.astype(float)).groupby("condition").m.mean() * 100).round(1).to_string())
    ck = d.dropna(subset=["constraint_kept"]).copy()
    if len(ck):
        ck["ck"] = ck.constraint_kept.astype(float)
        print("\nConstraint-kept / RQ2 rate by condition (%):")
        print((ck.groupby("condition").ck.mean() * 100).round(1).to_string())

try:
    subprocess.run([sys.executable, "-m", "analysis.analyze", "--config", CFG_PATH], cwd=REPO_DIR, check=True)
except Exception as e:
    print("\nanalysis.analyze may be limited on a tiny test set:", repr(e))

## Notes

- **Model:** `Qwen/Qwen2.5-3B-Instruct` (ungated, fits a T4). Change `MODEL_ID` in Cell 3 for any HF
  instruct model; larger ones may need 4-bit (`bitsandbytes`) or a bigger accelerator.
- **Pilot settings.** `LIMIT = 10` scenarios per kind, 2 repetitions, 6 turns, 3 clean repetitions, no judge
  (there is no independent judge model without API access; the judge is a secondary measure).
- **Not for the main experiment as is:** it generates one answer at a time (too slow for the full
  benchmark), the model judges itself, and the preregistered model size is 7–8B. The main run should
  serve the chosen 7–8B model with vLLM and send calls in parallel.
- **Sampling:** uses the model's own recommended settings (its `generation_config`), as preregistered;
  Cell 3 prints them so they can be recorded.
- **Checkpoint / resume:** results and the per-call cache live in `/kaggle/working/runs`. If the session
  ends, **Save Version**, then in a new version attach this notebook's output as an input dataset — Cell 1
  restores it and Cell 5 continues from where it stopped (finished calls are served from the cache).
- **Full analysis** (`analysis.analyze`) writes tables/figures/report to `runs/kaggle/analysis/`; some
  tables need more than a tiny test set, hence the `try/except`.